In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
from pyhydra.data_sources.rainfall import download_aemet_daily_data
from pyhydra.data_sources.rainfall import AEMETDownloader, AemetCSVLoader

# 🌦️ AEMET Daily Climate Data Downloader

## General Description

This tool downloads **daily meteorological observations** from the **AEMET OpenData API**
(Agencia Estatal de Meteorología, Spain).

#### 🆚 AEMET vs alternatives — when to use which?

| Source | Coverage | Stations | Quality | Best for |
|--------|----------|----------|---------|----------|
| **AEMET** | Spain only | ~800 (dense) | High (national QC) | Spain studies, long records, official data |
| **Meteostat** | Global | ~70k (sparse) | Moderate | Quick downloads, international stations |
| **OGIMET** | Global | WMO network | Variable | Sub-daily SYNOP, cloud/visibility |
| **ERA5** | Global | 0.25° grid | Reanalysis | Ungauged areas, spatial fields |

> **For Spain**: AEMET is the reference source. It provides the densest network,
> longest records (some stations back to 1900s), and official national quality control.

## 📋 Variables available

| Column | Description | Unit |
|--------|-------------|------|
| `prec` | Daily precipitation | mm |
| `tmax` | Daily maximum temperature | °C |
| `tmin` | Daily minimum temperature | °C |
| `tmed` | Daily mean temperature | °C |
| `velmedia` | Mean wind speed | m/s |
| `racha` | Maximum wind gust | m/s |
| `sol` | Sunshine duration | h |
| `presMax` / `presMin` | Max/min atmospheric pressure | hPa |
| `hrMedia` | Mean relative humidity | % |

## 🔑 Prerequisites

1. Register at https://opendata.aemet.es and obtain a free API key
2. Set `API_KEY` in the configuration cell below

## 🗺️ Two-step workflow

| Step | Tool | Action |
|------|------|--------|
| 1 | `download_aemet_daily_data()` | Downloads raw NetCDF files for ALL stations |
| 2 | `AEMETDownloader` widget | Interactive map to select stations and export CSV |
| 3 | `AemetCSVLoader` | Loads exported CSVs for analysis |

> ⚠️ Step 1 downloads data for **all ~800 Spanish stations at once** — this can take 30–60 minutes
> and produces several GB of NetCDF files. Run once and reuse the cached files.


In [ ]:
# === CONFIGURATION ===
API_KEY = 'YOUR_AEMET_API_KEY'   # Get yours at https://opendata.aemet.es
OUTPUT_DIR = '/workspace/data/aemet/'

# Skip download if API key is not configured
if API_KEY == 'YOUR_AEMET_API_KEY':
    print('[AEMET] API key not configured — set API_KEY above.')
    print('[AEMET] Get your key at: https://opendata.aemet.es')
else:
    download_aemet_daily_data(
        path_output=OUTPUT_DIR,
        api_key=API_KEY,
        start_date_str='2010-01-01T00:00:00UTC',
        end_date_str='2024-12-31T23:59:59UTC'
    )

[AEMET] API key not configured — set API_KEY above.
[AEMET] Get your key at: https://opendata.aemet.es


# 📊 Extracting Variables and Exporting to CSV

Once the AEMET NetCDF files are downloaded, the **AEMETDownloader widget** lets you:
- Select stations on an interactive map by clicking markers
- Choose variable, date range, and output folder
- Export time series directly to CSV compatible with pyhydra pipelines

Alternatively, use `AemetCSVLoader` directly if you already have CSVs.

> ⚠️ The widget requires the NetCDF files from Step 1 to be present in `netcdf_folder`.
> If the folder is empty, run the download cell first.


In [ ]:
netcdf_folder = '/workspace/data/aemet/'

In [ ]:
AEMETDownloader(netcdf_folder)

Error loading NetCDF sample: [Errno 2] No such file or directory: '/workspace/data/aemet/'


In [ ]:
loader = AemetCSVLoader('/workspace/data/aemet/')

In [ ]:
stations_df = loader.load_station_data()
stations_df

In [ ]:
try:
    series_df = loader.load_series_data('prec')
    series_df.head()
except Exception as e:
    print(f'[AEMET] Skipped (no data available): {e}')


[AEMET] Skipped (no data available): [Errno 2] No such file or directory: '/workspace/data/aemet/'


In [ ]:
try:
    quality_df = loader.analyze_series_quality()
    quality_df
except Exception as e:
    print(f'[AEMET] Skipped (no data available): {e}')


[AEMET] Skipped (no data available): Load data first with load_series_data().


---
## 📋 Quality metrics interpretation

`analyze_series_quality()` checks each downloaded station series:

| Column | Meaning | Threshold |
|--------|---------|----------|
| `missing_percent` | % of days with NaN | >5%: infill; >10%: exclude from extremes |
| `full_years` | Years with <1% missing | Need ≥10 for frequency analysis |
| `min_value` / `max_value` | Range check | Negative precipitation → data error |

#### ⚠️ AEMET-specific quality notes

- Some old stations have **instrument changes** that cause sudden shifts in mean values —
  look for step changes in the annual means plot
- Precipitation records marked `'Ip'` (inapreciable) represent trace amounts (<0.1 mm);
  they are converted to 0.0 in pyhydra
- Mountain stations (>1500 m) may have winter gaps due to snow covering the rain gauge

After quality control, series connect to:
- **Spatial interpolation** → `interpolation` notebook
- **Extreme value analysis** → `extreme_value_analysis` notebook
- **Stochastic generation** → `stochastic_generation` notebook
